In [13]:

import numpy as np
import pandas as pd
from iminuit import Minuit
from iminuit.cost import LeastSquares
from scipy.integrate import quad, fixed_quad
from scipy.special import j0
import plotly.graph_objects as go



In [14]:
# experimental data
save_folder = 'run7'
n_points = 5000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_23360/4146914037.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_23360/4146914037.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [15]:
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

atlas_pl_parameters = {
    'epsilon': 0.0753,
    'mg': 0.421,
    'a1': 1.517,
    'a2': 2.05,
}

ensemble_atlas = 'atlas'
pl_model_type = 'pl'

def get_atlas_pl_params():
    return atlas_pl_parameters

initial_params_pl_atlas = get_atlas_pl_params()


In [16]:
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(
        rho_mg_squared / lambda_squared
    )
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))


def alpha_D(q2, mg):
    m2 = m2_pl(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4.0 * m2) / (Lambda ** 2)))


def T_1(k, q2, phi, mg, a1, a2):
    sqrt_q2 = np.sqrt(q2)
    qk_cos = sqrt_q2 * k * np.cos(phi)

    qk_plus_squared = q2 / 4.0 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4.0 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg)
    alpha_D_minus = alpha_D(qk_minus_squared, mg)

    G0 = G_p(q2, a1, a2)

    return alpha_D_plus * alpha_D_minus * G0 ** 2


def T_2(k, q2, phi, mg, a1, a2):
    sqrt_q2 = np.sqrt(q2)
    qk_cos = sqrt_q2 * k * np.cos(phi)

    qk_plus_squared = q2 / 4.0 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4.0 - qk_cos + k ** 2

    alpha_D_plus = alpha_D(qk_plus_squared, mg)
    alpha_D_minus = alpha_D(qk_minus_squared, mg)

    factor = q2 + 9.0 * abs(k ** 2 - q2 / 4.0)

    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)

    return alpha_D_plus * alpha_D_minus * G_minus * (2.0 * G0 - G_minus)


def integrand(y, x, mg, a1, a2, q_val, sqrt_s):
    k = sqrt_s * x
    phi = 2.0 * np.pi * y
    jacobian = 2.0 * np.pi * sqrt_s

    T1 = T_1(k, q_val, phi, mg, a1, a2)
    T2 = T_2(k, q_val, phi, mg, a1, a2)

    return k * (T1 - T2) * jacobian


def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323


def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s ** alpha_pomeron) / (s0 ** (alpha_pomeron - 1.0))
    return 1j * 8.0 * regge_factor * diff_T


def differential_sigma(amp_value, s):
    amp_imag = amp_value.imag
    denominator = 16.0 * np.pi * s ** 2
    return (amp_imag * amp_imag) / denominator * 0.389379323


In [17]:
from scipy.integrate import quad, dblquad
import numpy as np

def full_int(mg, a1, a2, q2_val, sqrt_s):
    q2_val = np.atleast_1d(q2_val)
    results = []
    
    epsabs = 1e-6
    epsrel = 1e-4
    
    for q2 in q2_val:
        # Try transformation if integrand has issues near boundaries
        def integrand_transformed(u, v):
            # Transform to handle potential boundary issues
            # Using tanh transformation can help with endpoint singularities
            x = u  # Could use: 0.5 * (1 + np.tanh(np.pi * (u - 0.5)))
            y = v  # Could use: 0.5 * (1 + np.tanh(np.pi * (v - 0.5)))
            
            k = sqrt_s * x
            phi = 2.0 * np.pi * y
            jacobian = 2.0 * np.pi * sqrt_s
            
            T1 = T_1(k, q2, phi, mg, a1, a2)
            T2 = T_2(k, q2, phi, mg, a1, a2)
            
            return k * (T1 - T2) * jacobian
        
        # Simple Monte Carlo integration as fallback
        def monte_carlo_integral(n_samples=100000):
            total_real = 0.0
            total_imag = 0.0
            
            for _ in range(n_samples):
                x = np.random.random()
                y = np.random.random()
                k = sqrt_s * x
                phi = 2.0 * np.pi * y
                jacobian = 2.0 * np.pi * sqrt_s
                
                T1 = T_1(k, q2, phi, mg, a1, a2)
                T2 = T_2(k, q2, phi, mg, a1, a2)
                
                val = k * (T1 - T2) * jacobian
                total_real += np.real(val)
                total_imag += np.imag(val)
            
            area = 1.0  # Integration area is 1x1
            return (total_real / n_samples * area + 
                    1j * total_imag / n_samples * area)
        
        try:
            # Try dblquad first
            def real_integrand(y, x):
                return np.real(integrand_transformed(x, y))
            
            def imag_integrand(y, x):
                return np.imag(integrand_transformed(x, y))
            
            real_result, real_error = dblquad(real_integrand, 0.0, 1.0,
                                            lambda x: 0.0, lambda x: 1.0,
                                            epsabs=epsabs, epsrel=epsrel)
            
            imag_result, imag_error = dblquad(imag_integrand, 0.0, 1.0,
                                            lambda x: 0.0, lambda x: 1.0,
                                            epsabs=epsabs, epsrel=epsrel)
            
            result = real_result + 1j * imag_result
            
        except Exception as e:
            print(f"Integration failed for q2={q2}, using Monte Carlo: {e}")
            result = monte_carlo_integral()
        
        results.append(result)
    
    if len(results) > 1:
        return np.array(results)
    
    return results[0]

In [18]:
def model_function_powerlaw(x, eps, mg, a1, a2, sqrt_s):
    dif_sigma_lst = []

    s = sqrt_s ** 2

    for q2 in x:
        t = -q2

        integral_value = full_int(mg, a1, a2, q2, sqrt_s)
        amp_value = amp_calculation(integral_value, s, eps, t)
        dif_sigma_value = differential_sigma(amp_value, s)

        dif_sigma_lst.append(dif_sigma_value)

    return np.array(dif_sigma_lst)


def model_7(x, eps, mg, a1, a2):
    return model_function_powerlaw(x, eps, mg, a1, a2, sqrt_s=7000)


def model_8(x, eps, mg, a1, a2):
    return model_function_powerlaw(x, eps, mg, a1, a2, sqrt_s=8000)


def model_13(x, eps, mg, a1, a2):
    return model_function_powerlaw(x, eps, mg, a1, a2, sqrt_s=13000)


chi2_7 = LeastSquares(x_7_atlas, y_7_atlas, yerr_7_atlas, model_7)
chi2_8 = LeastSquares(x_8_atlas, y_8_atlas, yerr_8_atlas, model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)

chi2_total = chi2_7 + chi2_8 + chi2_13

minuit_born = Minuit(
    chi2_total,
    mg=0.421,
    a1=1.517,
    a2=2.05,
    eps=0.0753,
)

minuit_born.simplex()
minuit_born.migrad()
minuit_born.hesse()


/home/victorli/miniconda3/envs/two_gluon/lib/python3.11/site-packages/scipy/integrate/_quadpack_py.py:1264: IntegrationWarning: The algorithm does not converge.  Roundoff error is detected
  in the extrapolation table.  It is assumed that the requested tolerance
  cannot be achieved, and that the returned result (if full_output = 1) is 
  the best which can be obtained.
  quad_r = quad(f, low, high, args=args, full_output=self.full_output,


KeyboardInterrupt: 

In [ ]:
# Calculates and plot dif sigma 
lst_amp_born_diff = []
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001


    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2
        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)
        print(integral_value)
        
        diff_T = integral_value


        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        lst_amp_born_diff.append(amp_value)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    minuit_born.values["eps"],
    minuit_born.values["mg"],
    minuit_born.values["a1"],
    minuit_born.values["a2"],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


In [ ]:


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [ ]:
import numpy as np
from scipy.integrate import quad_vec
from scipy.special import j0

# -------------------------------------------------
# Fixed kinematics
# -------------------------------------------------
sqrt_s = 7000.0
s = sqrt_s ** 2

# -------------------------------------------------
# Impact parameter grid
# -------------------------------------------------
lst_b_integration = np.linspace(0.0, 20.0, 200)

# -------------------------------------------------
# Output
# -------------------------------------------------
lst_chi = []

# -------------------------------------------------
# χ(b) with convergence-controlled Hankel integral
# -------------------------------------------------
def chi_of_b(b_val):

    def integrand(q):
        q2 = q * q
        t = -q2

        diff_t = full_int(
            minuit_born.values['mg'],
            minuit_born.values['a1'],
            minuit_born.values['a2'],
            m2_pl,
            q2,
            sqrt_s
        )

        born_amp = amp_calculation(
            diff_t,
            s,
            minuit_born.values['eps'],
            t
        )

        return (q / s) * j0(b_val * q) * born_amp

    # --- adaptive extension ---
    q_edges = np.linspace(0.0, 10.0, 6)  # initial windows
    total = 0.0 + 0.0j

    for i in range(len(q_edges) - 1):
        val, _ = quad_vec(integrand, q_edges[i], q_edges[i+1])
        total += val

    # extend tail until converged
    q_start = q_edges[-1]
    step = 5.0

    for _ in range(10):
        val, _ = quad_vec(integrand, q_start, q_start + step)
        total += val

        if np.abs(val) < 1e-8 * np.abs(total):
            break

        q_start += step

    return total

# -------------------------------------------------
# Main loop
# -------------------------------------------------
for b_val in lst_b_integration:
    chi_val = chi_of_b(b_val)
    print(b_val, chi_val)
    lst_chi.append(chi_val)
